In [1]:
!pip install msal requests

In [ ]:
TENANT_ID =""
CLIENT_ID = ""
CLIENT_SECRET=""

In [3]:
import requests
from msal import ConfidentialClientApplication


AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["https://graph.microsoft.com/.default"]

app = ConfidentialClientApplication(
    CLIENT_ID,
    authority=AUTHORITY,
    client_credential=CLIENT_SECRET,
)

token_response = app.acquire_token_for_client(scopes=SCOPE)

access_token = token_response.get("access_token")

print("Token received:", access_token[:50])

Token received: eyJ0eXAiOiJKV1QiLCJub25jZSI6IkhFZlJJRk1leXYtcHhjTW


In [ ]:
site_url = "globalappsportal.sharepoint.com:/sites/DLQLeadQualificationOps"

url = f"https://graph.microsoft.com/v1.0/sites/{site_url}"

headers = {
    "Authorization": f"Bearer {access_token}"
}

response = requests.get(url, headers=headers)

site_data = response.json()

site_id = site_data["id"]

print("SITE ID:", site_id)



In [ ]:
url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/lists"

response = requests.get(url, headers=headers)

lists = response.json()

for l in lists["value"]:
    print(l["name"], "→", l["id"])

In [ ]:
LIST_ID = "your-list-id"

url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/lists/{LIST_ID}/items?expand=fields"

response = requests.get(url, headers=headers)

data = response.json()

items = data["value"]

for item in items:
    print(item["fields"])

In [ ]:
import pandas as pd

records = [item["fields"] for item in items]

df = pd.DataFrame(records)

df.head()

## another way

In [ ]:
from msal import PublicClientApplication
import requests

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPES = ["Sites.Read.All"]

app = PublicClientApplication(CLIENT_ID, authority=AUTHORITY)

flow = app.initiate_device_flow(scopes=SCOPES)

print(flow["message"])  # login instructions

result = app.acquire_token_by_device_flow(flow)

access_token = result["access_token"]